In [1]:
import os

import fsspec
import h5netcdf

import pyfive
import zarr

import xarray

In [2]:
os.environ["H5NETCDF_READ_BACKEND"] = "pyfive"

# native

### kerchunk (zarr)

In [9]:
z = zarr.open(
        fsspec.filesystem(
        "reference",
        fo="https://uor-aces-o.s3-ext.jc.rl.ac.uk/bnl/uas_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_185001-234912.nc_cmip7repack4mb.json", 
        target_protocol="https",
        target_options={"asynchronous": False}, 
        remote_protocol="https",
        remote_options={"asynchronous": True},
        asynchronous=True,
    ).get_mapper(),
    mode="r")

In [10]:
%time v = z["uas"]

CPU times: user 0 ns, sys: 3.65 ms, total: 3.65 ms
Wall time: 2.65 ms


In [11]:
%time a = v[...]

CPU times: user 5.75 s, sys: 1.93 s, total: 7.69 s
Wall time: 11.6 s


### pyfive

In [12]:
fs = fsspec.open("https://uor-aces-o.s3-ext.jc.rl.ac.uk/bnl/uas_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_185001-234912.nc_cmip7repack4mb")
f = pyfive.File(fs.open())

In [13]:
%time v = f["uas"]

CPU times: user 21.9 ms, sys: 235 μs, total: 22.1 ms
Wall time: 482 ms


In [14]:
%time a = v[...]

CPU times: user 5.74 s, sys: 1.89 s, total: 7.63 s
Wall time: 12.3 s


# xarray

### kerchunk

In [3]:
ds = xarray.open_dataset(
    "reference://",
    engine="zarr",
    backend_kwargs={
        "storage_options": {
                    "fo": "https://uor-aces-o.s3-ext.jc.rl.ac.uk/bnl/uas_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_185001-234912.nc_cmip7repack4mb.json",
        "target_protocol": "https",
        "target_options": {"asynchronous": False},
        "remote_protocol": "https",
        "remote_options": {"asynchronous": True},
        "asynchronous": True,
        },
        "consolidated": False,
    },
    decode_times=False)

In [4]:
%time v = ds["uas"]

CPU times: user 88 μs, sys: 0 ns, total: 88 μs
Wall time: 99.4 μs


In [5]:
%time _ = v[...].load()

CPU times: user 6.28 s, sys: 2.82 s, total: 9.1 s
Wall time: 15.8 s


### pyfive

In [6]:
ds = xarray.open_dataset(
    "https://uor-aces-o.s3-ext.jc.rl.ac.uk/bnl/uas_Amon_IPSL-CM6A-LR_piControl_r1i1p1f1_gr_185001-234912.nc_cmip7repack4mb",
    engine="h5netcdf",
    decode_times=False)

In [7]:
%time v = ds["uas"]

CPU times: user 122 μs, sys: 0 ns, total: 122 μs
Wall time: 136 μs


In [8]:
%time _ = v[...].load()

CPU times: user 6.04 s, sys: 2.48 s, total: 8.52 s
Wall time: 14.6 s
